# Gradient calculation in backpropagation

***

## Naive approach

Given a loss function $L$, we are interested in calculating the gradients of the loss function with respect to network parameters. These gradients are used to update network parameters via gradient descent.

Naive approach in this context refers to applying chain rule and derivative rules from standard calculus.

### Example - Dense Layer

For a fully connected DenseLayer with parameters $W \in \mathbb{R}^{n \times p}, b \in \mathbb{R}^{p}$, the transformation applied to input data $X \in \mathbb{R}^{m \times n}$ is the following:

$$ Y = XW + b $$

To update parameters $W, b$ we need to calculate gradients with respect to loss function $\frac{\partial L}{\partial W}$ and $\frac{\partial L}{\partial b}$. Suppose we obtain the upstream gradient from the network with respect to dense layer output $\delta = \frac{\partial L}{\partial Y} \in \mathbb{R}^{m \times p}$. 

Using the chain rule and naively supposing via derivative rules $\frac{\partial Y}{\partial W} = X, \frac{\partial Y}{\partial X} = W$ and $\frac{\partial Y}{\partial b}=1$ we multiply gradients:

$$\frac{\partial L}{\partial W} = \frac{\partial L}{\partial Y} \cdot \frac{\partial Y}{\partial W} = \delta \cdot X \quad \quad  \frac{\partial L}{\partial X} = \frac{\partial L}{\partial Y} \cdot \frac{\partial Y}{\partial X} = \delta \cdot W \quad \quad \frac{\partial L}{\partial b} = \frac{\partial L}{\partial Y} \cdot \frac{\partial Y}{\partial b} = \delta \cdot 1 = \delta$$

The current problem with gradients $\frac{\partial L}{\partial W}$ and $\frac{\partial L}{\partial X}$ is that the shapes in matrix multiplication don't match.

- gradient $\frac{\partial L}{\partial W}$ should be of shape $n \times p$

- gradient $\frac{\partial L}{\partial X}$ should be of shape $m \times n$

- matrix multiplications $\delta \cdot X$ and $\delta \cdot W$ are not aligned shapewise, therefore cannot be computed

We can align matrices for multiplication by rearranging the terms and obtain a reasonable result:

$$\frac{\partial L}{\partial W} = \frac{\partial L}{\partial Y} \cdot \frac{\partial Y}{\partial W} = X^T \cdot \delta \quad \quad  \frac{\partial L}{\partial X} = \frac{\partial L}{\partial Y} \cdot \frac{\partial Y}{\partial X} = \delta \cdot W^T $$

Conceptually, standard calculus chain rule and derivative rules can be a starting point when deriving gradients in deep learning. However, there are some drawbacks:

- classic chain rule does not account for matrix shapes and order of operations

- gradients such as $\frac{\partial Y}{\partial W}$ and $\frac{\partial Y}{\partial X}$ are in fact Jacobian 4D tensors of shape $(m \times p \times n \times p)$ and $(m \times p \times n \times p)$ respectively

- using 4D tensors in practice would be computationally and memory expensive

***

## More rigorous approach

- https://xiucheng.org/assets/pdfs/matrix-calculus4ml.pdf

In order to get more shape friendly and aligned gradients, we will utilize some matrix calculus rules.

### 1) Frobenius inner product

Similar to standard dot product between two vectors, **Frobenius inner product** is an extension to matrices. For matrices $A, B \in \mathbb{R}^{m \times n}$ the product is defined as

$$\langle A, B \rangle_F = \text{tr}(A^T B) = \sum_{i=1}^m \sum_{j=1}^n A_{ij}B_{ij}$$

where $\text{tr}()$ is the trace operator which sums diagonal elements. For gradient derivation the trace version will prove most useful.

### 2) Trace operator properties

    2.1) Cyclic property

For matrices $A \in R^{a \times b}, B \in R^{b \times c}, C \in R^{c \times a}$

$$\text{tr}(ABC) = \text{tr}(C A B) = \text{tr}(B C A)$$

    2.2) Transpose property

$$\text{tr}(A^T) = \text{tr}(A)$$

    2.3) Linearity

$$\text{tr}(A + B) = \text{tr}(A) + \text{tr}(B)$$

### 3) Total differential

Total differential $dL$ of the loss function with respect to network parameters $\theta$ can be written using the Frobenius inner product as follows

$$dL = \langle \frac{\partial L}{\partial \theta}, d\theta \rangle_F = \text{tr} \left( \left ( \frac{\partial L}{\partial \theta} \right)^T d\theta \right)$$

This notation aids in determining the gradient of the loss function with respect to network parameters and is shape consistent.

**NOTE:** Total differential proves useful only when dealing with gradients of parameters that are involved in matrix multiplication, such as Dense Layer, Attention etc.

### Example - Dense Layer

We wish to find the expressions for $\frac{\partial L}{\partial W}$ and $\frac{\partial L}{\partial X}$ for the linear transformation $Y = WX + b$. We are provided an upstream gradient $\frac{\partial L}{\partial Y} = \delta$. Using the total differential and trace properties we will read off the gradient expressions when expanding on the differential $dY$. Using differential properties we express $dY$ as the following sum

$$dY = dXW + XdW + db$$

Using the obtained differential and trace properties we can start manipulating the $dL$ differential

$$
\begin{align*}
dL &= \text{tr} \left( \left ( \frac{\partial L}{\partial Y} \right)^T dY \right) \\
&= \text{tr} \left( \delta^T (dXW + XdW + db) \right) \\
&= \text{tr} \left(\delta^TdXW + \delta^TXdW + \delta^Tdb  \right) \\
&= \text{tr} \left(\delta^TdXW \right) + \text{tr} \left( \delta^TXdW \right) + \text{tr} \left( \delta^Tdb \right) \\
&= \text{tr} \left(W\delta^TdX \right) + \text{tr} \left( \delta^TXdW \right) + \text{tr} \left( \delta^Tdb \right) \\
\end{align*}
$$

Closely observing the terms that are to the left of differentials $dX$, $dW$ and $db$ we can read off the gradients.

$$
\begin{align*}
\text{tr} \left(W\delta^TdX \right) &\Rightarrow \frac{\partial L}{\partial X}^T = W\delta^T \Rightarrow \frac{\partial L}{\partial X} = \delta W^T \\
\text{tr} \left( \delta^TXdW \right) &\Rightarrow \frac{\partial L}{\partial W}^T = \delta^TX \Rightarrow \frac{\partial L}{\partial W} = X^T \delta \\
\text{tr} \left( \delta^Tdb \right) &\Rightarrow \frac{\partial L}{\partial b}^T = \delta^T \Rightarrow \frac{\partial L}{\partial b} = \delta
\end{align*}
$$

***

### Takeaways

- naive approach using single variable calculus intuition is limited when dealing with matrices

- determining gradients of activation functions which act element-wise on the matrices can be done using single variable calculus formulas

- finding gradients of parameters involved in matrix products requires the use of total differential and trace operator